In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score

# 1. SETUP PATHS
# We use absolute paths derived from the script location to avoid FileNotFoundError
BASE_DIR = os.path.dirname(os.getcwd()) # Goes from notebooks/ up to ml/
DATA_PATH = os.path.join(os.path.dirname(BASE_DIR), 'data', 'air_quality_india.csv')
MODEL_DIR = os.path.join(BASE_DIR, 'models')

if not os.path.exists(MODEL_DIR):
    os.makedirs(MODEL_DIR)

# 2. LOAD DATA
print(f"Loading data from: {DATA_PATH}")
df = pd.read_csv(DATA_PATH)

# 3. PREPROCESSING
# Select only relevant columns
cols_to_use = ['City', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI']
df = df[cols_to_use]

# Fill missing values with median (Production best practice)
for col in df.columns:
    if df[col].dtype in ['float64', 'int64']:
        df[col] = df[col].fillna(df[col].median())

# Encode City names into numbers
encoder = LabelEncoder()
df['City_Encoded'] = encoder.fit_transform(df['City'])

# 4. SPLIT DATA
features = ['City_Encoded', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']
X = df[features]
y = df['AQI']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. TRAIN MODELS
print("Training Random Forest...")
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

print("Training Linear Regression (Baseline)...")
lr = LinearRegression()
lr.fit(X_train, y_train)

# 6. EVALUATION
rf_preds = rf.predict(X_test)
lr_preds = lr.predict(X_test)

print("-" * 30)
print(f"RF Mean Absolute Error: {mean_absolute_error(y_test, rf_preds):.2f}")
print(f"RF R2 Score: {r2_score(y_test, rf_preds):.2f}")
print(f"LR Mean Absolute Error: {mean_absolute_error(y_test, lr_preds):.2f}")
print("-" * 30)

# 7. SAVE ARTIFACTS
joblib.dump(rf, os.path.join(MODEL_DIR, 'random_forest.pkl'))
joblib.dump(lr, os.path.join(MODEL_DIR, 'linear_regression.pkl'))
joblib.dump(encoder, os.path.join(MODEL_DIR, 'city_encoder.pkl'))

print(f"✅ Success! All models saved in: {MODEL_DIR}")

Loading data from: c:\Users\Abhyuday\Documents\AirQuality\data\air_quality_india.csv
Training Random Forest...
Training Linear Regression (Baseline)...
------------------------------
RF Mean Absolute Error: 19.88
RF R2 Score: 0.88
LR Mean Absolute Error: 29.86
------------------------------
✅ Success! All models saved in: c:\Users\Abhyuday\Documents\AirQuality\ml\models
